# McMaster browse (optional workflow)

Separate from the MakerWorld pipeline (`01`–`06`). Exercises in-house browse scrape + parse.

| | |
|---|---|
| **Matcher** | `matcher.match_part` + `guess_strategy.py` (primary / same_size / wider_scope) |
| **Scrape** | `backend.services.vendors.mcmaster.browse_scrape` (Playwright ProdPageWebPart) |
| **Parse** | `backend.services.vendors.mcmaster.browse_parse` |
| **Row pick** | `browse_row_select.pick_simplest_browse_row` (primary SKU; default hydration policy) |
| **Fetch** | `backend.services.vendors.mcmaster.browse_fetch` (gated by `MCMASTER_BROWSE_RESOLVE_ENABLED=1`) |
| **Archive reference** | `docs/archive/mcmaster-scraper-v0.2.1/` (upstream v0.2.1 sanity check) |
| **Demo script** | `scripts/mcmaster_browse_example.py` |

**Offline:** fixture cell (no network). **Live:** set `MCMASTER_BROWSE_RESOLVE_ENABLED=1` + Playwright (`pip install -e '.[playwright]' && playwright install chromium`).


In [ ]:
import os

from backend.notebook_utils import prepare_crawl_env

prepare_crawl_env(scraper="auto")
os.environ.setdefault("MCMASTER_BROWSE_RESOLVE_ENABLED", "0")

## Offline — parse fixture JSON (no network)

In [ ]:
import json
from pathlib import Path

from backend.notebook_utils import browse_rows_to_dataframe
from backend.services.vendors.mcmaster.browse_parse import parse_product_presentations

fixture = Path("../tests/fixtures/mcmaster_product_presentations_min.json")
payload = json.loads(fixture.read_text())
rows = parse_product_presentations(payload)
browse_rows_to_dataframe(rows)

## Offline — filtered browse URLs from matcher (structured guesses)

Primary = size-filtered table. Secondaries split into `same_size` (other materials) and `wider_scope` (broader browse).

In [ ]:
from backend.models.part import Part
from backend.notebook_utils import alternatives_to_dataframe, parts_to_dataframe
from backend.services.matcher import match_part

DEMO_LINES = [
    "M3x16 socket head cap screw",
    "M3 Nut",
    "M3 nylock",
    "M4 flat washer",
    '#6-32 x 1/2"',
]

matched_parts = [match_part(Part(original_name=name)) for name in DEMO_LINES]
display(parts_to_dataframe(matched_parts))

for part in matched_parts:
    print(f"\n=== {part.original_name} ===")
    print(part.match_tier, part.mcmaster_category, f"{part.confidence:.0%}")
    print(part.mcmaster_url[:120], "…")
    if part.mcmaster_reason:
        print(part.mcmaster_reason[:160])

display(alternatives_to_dataframe(matched_parts))

## Offline — simplest row pick (fixture table)

When hydration runs, the primary SKU uses `pick_simplest_browse_row` unless `match_selection_policy` is `lowest_price`.

In [ ]:
from backend.models.part import Part
from backend.services.vendors.mcmaster.browse_parse import BrowseRow
from backend.services.vendors.mcmaster.browse_row_select import pick_simplest_browse_row

part = Part(original_name="M3 Nut")
rows = [
    BrowseRow(part_number="90591A100", product_type="Hex Nut", fields={"Thread Size": "M3"}),
    BrowseRow(
        part_number="90591A200",
        product_type="Nylon-Insert Locknut",
        fields={"Thread Size": "M3"},
    ),
]
picked = pick_simplest_browse_row(part, rows)
print("Picked:", picked.part_number if picked else None)

## Live — fetch browse table (Playwright)

Uncomment and run when Playwright is installed. Respect McMaster rate limits.

In [ ]:
# import os
# os.environ["MCMASTER_BROWSE_RESOLVE_ENABLED"] = "1"
#
# from backend.notebook_utils import browse_rows_to_dataframe
# from backend.services.vendors.mcmaster.browse_fetch import fetch_browse_rows
#
# url = matched.mcmaster_url
# rows = await fetch_browse_rows(url)
# browse_rows_to_dataframe(rows).head(10)

## Cross-test — matcher vs pipeline

Curated cases in `data/mcmaster_regression_urls.json`. Same checks as `scripts/mcmaster_cross_test.py` (also in `run_checks.sh`).

In [ ]:
from backend.notebook_utils import load_mcmaster_regression_catalog, run_mcmaster_cross_test_offline

catalog = load_mcmaster_regression_catalog()
print(f"{len(catalog['cases'])} curated cases\n")
print(run_mcmaster_cross_test_offline())

## Live cross-test (Playwright)

Uncomment to fetch browse tables for filtered-browse cases and verify expected SKUs appear.

In [ ]:
# import asyncio
# import os
#
# os.environ["MCMASTER_BROWSE_RESOLVE_ENABLED"] = "1"
#
# from backend.services.vendors.mcmaster.cross_test import run_live_cross_test, format_cross_test_report
#
# live_results = asyncio.run(run_live_cross_test())
# print(format_cross_test_report(live_results))

## CLI demos

```bash
python scripts/mcmaster_browse_example.py --offline   # fixture parse
MCMASTER_BROWSE_RESOLVE_ENABLED=1 python scripts/mcmaster_browse_example.py
python scripts/mcmaster_cross_test.py               # offline cross-test
python scripts/mcmaster_cross_test.py --live        # live cross-test
```